# FLEX — Daily Log-Return Forecasting & Quantile Trading

**Course:** APS1052 — AI in Finance &nbsp;|&nbsp; **Team:** Yicheng Yao, Jarvis Wang, JiangChuan Yu

This notebook builds an end-to-end machine-learning / deep-learning pipeline that predicts the
**next-day log return of FLEX (Flex Ltd.)** and trades it with a quantile-based strategy,
evaluated with anti–data-snooping tests.

## Design decisions (see `../project_decisions.txt`)

| # | Decision | Choice |
|---|----------|--------|
| 1 | Target asset | **FLEX** (mid-cap, Information Technology) |
| 2 | Target variable | **Next-day log return** (regression) |
| 3 | Frequency | **Daily** |
| 4 | Targets | **Single-target** |
| 5 | Data source | **Yahoo Finance + FRED** (free) |
| 6 | Models (>=5, >=1 scaled) | Ridge, Random Forest, LightGBM, **SVR**, **Keras MLP** |
| 7 | Indicators | **18** total, 12 (67%) NOT from target's OHLCV bar |
| 8 | Custom indicator | **Relative strength vs. sector** (hand-coded) |
| 9 | Feature selection | `SelectKBest` (mutual info); **discard worst 4** |
| 10 | Trading | Long top quantile (Q5), short bottom (Q1), flat otherwise |
| 11 | Scaling | **Hybrid**: rolling (price/return/macro) + independent-row (Ta-Lib) + encode (calendar) |
| 12 | Input processing | Fractional differencing + return transforms + light causal smoothing of macro |
| 13 | Hyperparameter search | Manual grid-search CV loop with `TimeSeriesSplit` |
| 14 | Neural net | MSE loss, L2 + dropout, no sample weights |
| 16 | Split | Chronological ~70/15/15 |

**Pipeline:** data -> features -> hybrid scaling -> split -> feature selection -> model selection ->
fine-tuning -> metrics -> SHAP -> trading & equity curve -> White / Monte-Carlo diagnostics.

> **No PyTorch is used** (course rule). The neural network is a Keras MLP (TensorFlow backend).

## 0. Setup, configuration and reproducibility

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_selection import SelectKBest, mutual_info_regression
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error
from scipy.stats import spearmanr

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ----------------------------------------------------------------------------
# Configuration
# ----------------------------------------------------------------------------
TARGET      = "FLEX"      # mid-cap, Information Technology
SECTOR_ETF  = "XLK"       # Information Technology sector ETF
START       = "2010-01-01"
END         = "2025-06-01"
DATA_DIR    = "data"
IMG_DIR     = "images"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(IMG_DIR, exist_ok=True)

# Chronological split fractions (no shuffling for time series)
TRAIN_FRAC, VAL_FRAC = 0.70, 0.15   # test = remaining 0.15
N_QUANTILES = 5                     # quintiles for the trading strategy

# ----------------------------------------------------------------------------
# Optional dependencies: LightGBM and SHAP fall back gracefully if missing.
# ----------------------------------------------------------------------------
try:
    from lightgbm import LGBMRegressor
    HAS_LGBM = True
except Exception:
    HAS_LGBM = False
    print("LightGBM not found -> falling back to sklearn GradientBoostingRegressor.")

try:
    import shap
    HAS_SHAP = True
except Exception:
    HAS_SHAP = False
    print("SHAP not found -> falling back to sklearn permutation_importance.")

try:
    from fracdiff import fdiff  # noqa: F401
    HAS_FRACDIFF = True
except Exception:
    HAS_FRACDIFF = False  # we provide a manual De Prado fixed-width implementation

try:
    import tensorflow as _tf  # noqa: F401
    HAS_TF = True
except Exception:
    HAS_TF = False
    print("TensorFlow not found -> the Keras MLP will be skipped (install tensorflow to include it).")

print("Setup complete. Target =", TARGET, "| Sector ETF =", SECTOR_ETF)

## 1. Data acquisition (Yahoo Finance + FRED)

All series are downloaded once and cached to `data/*.csv` so the project is fully
reproducible and the data can be submitted with the notebook (course requirement).
On later runs the cached CSVs are reused (no network needed).

In [ ]:
# ----------------------------------------------------------------------------
# Robust downloaders.
# Newer yfinance (>=1.x) uses curl_cffi and can fail on some machines with
# "SSL certificate problem: unable to get local issuer certificate".
# We therefore hit Yahoo's public chart API directly through curl_cffi using
# the certifi CA bundle, and fall back to yfinance only if that fails.
# ----------------------------------------------------------------------------
import time
import datetime as _dt
import io as _io
import certifi

def _cffi_session():
    from curl_cffi import requests as _cr
    return _cr.Session(impersonate="chrome", verify=certifi.where())


def _yahoo_chart(ticker, start, end, interval="1d"):
    p1 = int(time.mktime(_dt.datetime.strptime(start, "%Y-%m-%d").timetuple()))
    p2 = int(time.mktime(_dt.datetime.strptime(end, "%Y-%m-%d").timetuple()))
    url = f"https://query1.finance.yahoo.com/v8/finance/chart/{ticker}"
    params = {"period1": p1, "period2": p2, "interval": interval, "events": "div,splits"}
    r = _cffi_session().get(url, params=params, timeout=30)
    r.raise_for_status()
    res = r.json()["chart"]["result"][0]
    idx = pd.to_datetime(res["timestamp"], unit="s").normalize()
    q = res["indicators"]["quote"][0]
    df = pd.DataFrame({"Open": q["open"], "High": q["high"], "Low": q["low"],
                       "Close": q["close"], "Volume": q["volume"]}, index=idx)
    adj = res["indicators"].get("adjclose")
    if adj:  # mimic auto_adjust=True: scale OHLC by adjclose/close
        adjclose = pd.Series(adj[0]["adjclose"], index=idx)
        factor = (adjclose / df["Close"]).replace([np.inf, -np.inf], np.nan).fillna(1.0)
        for c in ["Open", "High", "Low", "Close"]:
            df[c] = df[c] * factor
    df.index.name = "Date"
    return df.dropna(how="all")


def get_yahoo(ticker, fname, start=START, end=END):
    """Download daily OHLCV (adjusted), cache to CSV, return a clean DataFrame."""
    path = os.path.join(DATA_DIR, fname)
    if os.path.exists(path):
        return pd.read_csv(path, index_col=0, parse_dates=True)
    try:
        raw = _yahoo_chart(ticker, start, end)
    except Exception as e:
        print(f"  chart API failed for {ticker} ({e}); trying yfinance...")
        import yfinance as yf
        raw = yf.download(ticker, start=start, end=end, interval="1d",
                          auto_adjust=True, progress=False)
        if isinstance(raw.columns, pd.MultiIndex):
            raw.columns = raw.columns.get_level_values(0)
    if raw is None or raw.empty:
        raise RuntimeError(f"No data returned for {ticker}")
    raw.index.name = "Date"
    raw.to_csv(path)
    return raw


def get_fred(series_id, fname):
    """Download a single FRED series via its public CSV endpoint, cache to CSV."""
    path = os.path.join(DATA_DIR, fname)
    if os.path.exists(path):
        s = pd.read_csv(path, index_col=0, parse_dates=True)
        return pd.to_numeric(s.iloc[:, 0], errors="coerce")
    url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}"
    try:
        df = pd.read_csv(_io.StringIO(_cffi_session().get(url, timeout=30).text))
    except Exception:
        df = pd.read_csv(url)
    df.columns = ["Date", series_id]
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.set_index("Date")
    df[series_id] = pd.to_numeric(df[series_id], errors="coerce")
    df.to_csv(path)
    return df[series_id]


# --- Yahoo Finance series ---------------------------------------------------
flex = get_yahoo(TARGET,     f"{TARGET}.csv")
xlk  = get_yahoo(SECTOR_ETF, f"{SECTOR_ETF}.csv")
spy  = get_yahoo("SPY",      "SPY.csv")
vix  = get_yahoo("^VIX",     "VIX.csv")
move = get_yahoo("^MOVE",    "MOVE.csv")
hyg  = get_yahoo("HYG",      "HYG.csv")
ief  = get_yahoo("IEF",      "IEF.csv")

# --- FRED macro series ------------------------------------------------------
t10y2y = get_fred("T10Y2Y", "T10Y2Y.csv")   # 10Y-2Y yield-curve spread (recession indicator)
credit = get_fred("BAA10Y", "BAA10Y.csv")   # Baa corporate - 10Y Treasury credit spread (long history)

print("FLEX rows:", len(flex), "| range:", flex.index.min().date(), "->", flex.index.max().date())
flex.tail(3)

### 1.1 Target visualization

The target is the **next-day log return** of FLEX. Below we show the price level (for
context) and the daily log-return series the model will actually predict.

In [ ]:
flex_close = flex["Close"].astype(float)
flex_logret = np.log(flex_close).diff()

fig, ax = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
ax[0].plot(flex_close.index, flex_close, color="navy", lw=1)
ax[0].set_title(f"{TARGET} adjusted close price")
ax[0].set_ylabel("Price ($)")
ax[1].plot(flex_logret.index, flex_logret, color="darkred", lw=0.6)
ax[1].set_title(f"{TARGET} daily log return (target = this series shifted to t+1)")
ax[1].set_ylabel("log return")
plt.tight_layout()
plt.savefig(os.path.join(IMG_DIR, "target_overview.png"), dpi=120)
plt.show()

print(flex_logret.describe())

## 2. Feature engineering — 18 indicators

We build **18 indicators**. Per the course rule, **at least half must NOT come from the
target's own OHLCV bar** — here **12 of 18 (67%)** are external.

**Group A — from FLEX's own OHLCV bar (6):** RSI, MACD, ATR, Bollinger-band position,
historical log returns (1d + 5d), PPO.

**Group B — NOT from FLEX's OHLCV bar (12):** sector (XLK) return, SPY return, VIX, MOVE,
**relative strength vs. sector (custom)**, yield-curve spread (T10Y2Y), credit spread
(BAA10Y: Baa corporate − 10Y Treasury), risk-appetite ratio (HYG/IEF), De Prado
fractional-differenced close, day-of-week, month (cyclical), holiday proximity.

The helper functions below implement the Ta-Lib-style indicators and the De Prado
fixed-width fractional differencing directly in pandas/numpy (no compiled Ta-Lib dependency).

In [ ]:
# ---------------------------------------------------------------------------
# Ta-Lib-style technical indicators (pure pandas/numpy implementations)
# ---------------------------------------------------------------------------
def rsi(close, n=14):
    delta = close.diff()
    gain = delta.clip(lower=0).rolling(n).mean()
    loss = (-delta.clip(upper=0)).rolling(n).mean()
    rs = gain / loss.replace(0, np.nan)
    return 100 - 100 / (1 + rs)


def macd_line(close, fast=12, slow=26):
    ema_fast = close.ewm(span=fast, adjust=False).mean()
    ema_slow = close.ewm(span=slow, adjust=False).mean()
    return ema_fast - ema_slow


def atr(high, low, close, n=14):
    prev = close.shift()
    tr = pd.concat([(high - low), (high - prev).abs(), (low - prev).abs()], axis=1).max(axis=1)
    return tr.rolling(n).mean()


def bollinger(close, n=20, k=2):
    ma = close.rolling(n).mean()
    sd = close.rolling(n).std()
    return ma + k * sd, ma - k * sd


def ppo(close, fast=12, slow=26):
    ema_fast = close.ewm(span=fast, adjust=False).mean()
    ema_slow = close.ewm(span=slow, adjust=False).mean()
    return (ema_fast - ema_slow) / ema_slow * 100.0


# ---------------------------------------------------------------------------
# De Prado fixed-width-window fractional differencing (stationary + memory)
# ---------------------------------------------------------------------------
def fracdiff_ffd(series, d=0.4, thres=1e-4):
    w = [1.0]
    k = 1
    while True:
        w_ = -w[-1] * (d - k + 1) / k
        if abs(w_) < thres:
            break
        w.append(w_)
        k += 1
    w = np.array(w[::-1])
    width = len(w) - 1
    vals = series.astype(float).values
    out = np.full(len(vals), np.nan)
    for i in range(width, len(vals)):
        window = vals[i - width: i + 1]
        if not np.isnan(window).any():
            out[i] = np.dot(w, window)
    return pd.Series(out, index=series.index)


print("Indicator helper functions defined.")

In [ ]:
# Helper: align an external daily series onto FLEX trading days.
def align(series, how_fill="ffill"):
    s = series.reindex(flex.index)
    return s.ffill() if how_fill == "ffill" else s

H = flex["High"].astype(float)
L = flex["Low"].astype(float)
C = flex_close

xlk_ret = np.log(align(xlk["Close"].astype(float))).diff()
spy_ret = np.log(align(spy["Close"].astype(float))).diff()
vix_lvl = align(vix["Close"].astype(float))
move_lvl = align(move["Close"].astype(float))
hyg_c = align(hyg["Close"].astype(float))
ief_c = align(ief["Close"].astype(float))

# Light, causal smoothing (trailing window only) for noisy macro series.
t10y2y_s = align(t10y2y).rolling(5, min_periods=1).mean()
credit_s = align(credit).rolling(5, min_periods=1).mean()

# ----- Custom indicator: relative strength of FLEX vs its sector (XLK) -------
#   rs = FLEX log return - sector log return, smoothed over 10 days.
#   (z-scoring happens in the hybrid-scaling step via rolling scaling.)
rel_strength = (flex_logret - xlk_ret).rolling(10).mean()

bb_up, bb_lo = bollinger(C)

raw = pd.DataFrame(index=flex.index)
# --- Group A: from FLEX's own OHLCV bar (Ta-Lib style) ---
raw["rsi"]        = rsi(C)
raw["macd"]       = macd_line(C)
raw["atr"]        = atr(H, L, C)
raw["bb_pos"]     = 2 * (C - bb_lo) / (bb_up - bb_lo) - 1
raw["ret_1d"]     = flex_logret
raw["ret_5d"]     = np.log(C).diff(5)
raw["ppo"]        = ppo(C)
# --- Group B: NOT from FLEX's OHLCV bar ---
raw["xlk_ret"]    = xlk_ret
raw["spy_ret"]    = spy_ret
raw["vix"]        = vix_lvl
raw["move"]       = move_lvl
raw["rel_str"]    = rel_strength            # <-- custom indicator
raw["t10y2y"]       = t10y2y_s
raw["credit_spread"] = credit_s
raw["risk_ratio"] = np.log(hyg_c / ief_c)   # HYG/IEF risk-appetite ratio (log level)
raw["fracdiff"]   = fracdiff_ffd(np.log(C), d=0.4)
# calendar / categorical
raw["dow"]        = raw.index.dayofweek
raw["month"]      = raw.index.month

# Holiday proximity: trading days until the next US federal holiday.
from pandas.tseries.holiday import USFederalHolidayCalendar
hol = USFederalHolidayCalendar().holidays(start=START, end=END)
hol_arr = np.array(sorted(hol.values))
idx_arr = raw.index.values.astype("datetime64[ns]")
pos = np.searchsorted(hol_arr, idx_arr, side="left")
pos = np.clip(pos, 0, len(hol_arr) - 1)
days_to_hol = (hol_arr[pos] - idx_arr).astype("timedelta64[D]").astype(float)
raw["days_to_holiday"] = days_to_hol

print("Raw feature matrix:", raw.shape)
print("Columns:", list(raw.columns))
raw.tail(3)

## 3. Hybrid scaling

Per decision #11 we scale by **feature type** (the course warns that a single global
`StandardScaler` goes stale on later data and that rolling-scaling Ta-Lib features destroys
their meaning):

- **Independent-row scaling** for Ta-Lib features (no window, no train set needed):
  `RSI -> (RSI-50)/50`, `ATR -> log1p(ATR/Close)`, `MACD -> sign·log1p(|MACD/Close|)`,
  `PPO -> PPO/100`, BB position is already in ~[-1, 1].
- **Rolling scaling** (`asymmetric_rolling_scale`) for price/return/macro series.
- **Encoding** for calendar features: day-of-week one-hot, month as cyclical sin/cos,
  holiday proximity left as a small bounded count.

The rolling scaler is applied to the **whole series before splitting** (it is causal — it only
ever uses a trailing window — so this does not leak future information).

In [ ]:
def asymmetric_rolling_scale(series, mean_window=10, std_window=90, ddof=0, eps=1e-12):
    """scaled[t] = (x[t] - short_rolling_mean[t]) / long_rolling_std[t]. Causal."""
    series = series.astype(float)
    rmean = series.rolling(mean_window, min_periods=mean_window).mean()
    rstd = series.rolling(std_window, min_periods=std_window).std(ddof=ddof)
    rstd = rstd.where(rstd > eps, np.nan)
    return (series - rmean) / rstd


feat = pd.DataFrame(index=raw.index)

# --- Independent-row scaling for Ta-Lib features ---
feat["rsi"]    = (raw["rsi"] - 50) / 50
feat["macd"]   = np.sign(raw["macd"] / C) * np.log1p((raw["macd"] / C).abs())
feat["atr"]    = np.log1p(raw["atr"] / C)
feat["bb_pos"] = raw["bb_pos"]
feat["ppo"]    = raw["ppo"] / 100.0

# --- Rolling scaling for price/return/macro series ---
for col in ["ret_1d", "ret_5d", "xlk_ret", "spy_ret", "vix", "move",
            "rel_str", "t10y2y", "credit_spread", "risk_ratio", "fracdiff"]:
    feat[col] = asymmetric_rolling_scale(raw[col])

# --- Calendar / categorical encodings ---
dow_dummies = pd.get_dummies(raw["dow"].astype(int), prefix="dow").astype(float)
feat = feat.join(dow_dummies)
feat["month_sin"] = np.sin(2 * np.pi * raw["month"] / 12)
feat["month_cos"] = np.cos(2 * np.pi * raw["month"] / 12)
feat["days_to_holiday"] = raw["days_to_holiday"] / 21.0   # scale ~ months

# ----------------------------------------------------------------------------
# Target: NEXT-day log return. Features at t predict the return from t to t+1.
# ----------------------------------------------------------------------------
y = flex_logret.shift(-1)
y.name = "target_logret_t+1"

data = feat.join(y).dropna()
X = data.drop(columns=[y.name])
Y = data[y.name]
print("Final aligned dataset:", X.shape, "| target:", Y.shape)
X.tail(3)

## 4. Chronological train / validation / test split

Time series must be split **in time order** (no shuffling). Train ≈70%, validation ≈15%
(for model selection & tuning), test ≈15% (touched only once for final reporting).
`TimeSeriesSplit` is used *within* the training portion for cross-validation.

In [ ]:
n = len(X)
i_train = int(n * TRAIN_FRAC)
i_val = int(n * (TRAIN_FRAC + VAL_FRAC))

X_train, Y_train = X.iloc[:i_train], Y.iloc[:i_train]
X_val,   Y_val   = X.iloc[i_train:i_val], Y.iloc[i_train:i_val]
X_test,  Y_test  = X.iloc[i_val:], Y.iloc[i_val:]

# Returns actually realised over each period (for trading / equity curves later).
ret_train, ret_val, ret_test = Y_train.copy(), Y_val.copy(), Y_test.copy()

print(f"Train: {X_train.shape[0]} rows  {X_train.index.min().date()} -> {X_train.index.max().date()}")
print(f"Val  : {X_val.shape[0]} rows  {X_val.index.min().date()} -> {X_val.index.max().date()}")
print(f"Test : {X_test.shape[0]} rows  {X_test.index.min().date()} -> {X_test.index.max().date()}")

## 5. Feature selection — `SelectKBest` (mutual information), discard the worst 4

We score features by **mutual information** with the target on the **training set only**
(to avoid look-ahead). Following the course guidance that categorical sets are kept or dropped
together, the one-hot day-of-week columns and the month sin/cos pair are scored as **groups**.
We then **discard the 4 weakest** of the 18 logical indicators and keep the rest.

In [ ]:
# Map each of the 18 logical indicators to its column(s).
dow_cols = [c for c in X.columns if c.startswith("dow_")]
groups = {
    "rsi": ["rsi"], "macd": ["macd"], "atr": ["atr"], "bb_pos": ["bb_pos"],
    "ret_1d": ["ret_1d"], "ret_5d": ["ret_5d"], "ppo": ["ppo"],
    "xlk_ret": ["xlk_ret"], "spy_ret": ["spy_ret"], "vix": ["vix"], "move": ["move"],
    "rel_str": ["rel_str"], "t10y2y": ["t10y2y"], "credit_spread": ["credit_spread"],
    "risk_ratio": ["risk_ratio"], "fracdiff": ["fracdiff"],
    "day_of_week": dow_cols,
    "month": ["month_sin", "month_cos"],
    "days_to_holiday": ["days_to_holiday"],
}

# Mutual information on the TRAINING set only.
mi = mutual_info_regression(X_train.values, Y_train.values, random_state=RANDOM_STATE)
mi = pd.Series(mi, index=X_train.columns)

# Aggregate column scores into logical-indicator scores (max within a group).
group_score = {g: mi[cols].max() for g, cols in groups.items()}
group_score = pd.Series(group_score).sort_values(ascending=False)
print("Mutual-information ranking of the 18 indicators:\n")
print(group_score.round(5))

DISCARD_N = 4
dropped = list(group_score.tail(DISCARD_N).index)
kept_groups = [g for g in group_score.index if g not in dropped]
selected_cols = [c for g in kept_groups for c in groups[g]]

print(f"\nDiscarded worst {DISCARD_N}: {dropped}")
print(f"Kept {len(kept_groups)} indicators -> {len(selected_cols)} model columns.")

X_train_s = X_train[selected_cols]
X_val_s   = X_val[selected_cols]
X_test_s  = X_test[selected_cols]

## 6. Model selection — manual grid-search CV with `TimeSeriesSplit`

We try **5 models** (≥1 requiring scaling — both SVR and the MLP do): Ridge, Random Forest,
LightGBM (or sklearn GradientBoosting fallback), SVR, and a Keras MLP. For each model we run a
**manual grid-search cross-validation loop** using `TimeSeriesSplit` on the training data,
scoring by validation MSE, then refit the best configuration and compare models on the
held-out **validation** set using the full metric suite (MAE, Profit Factor, Spearman RHO,
directional accuracy).

First we define the shared evaluation functions used throughout the rest of the notebook.

In [ ]:
# ---------------------------------------------------------------------------
# Shared evaluation + trading helper functions
# ---------------------------------------------------------------------------
def profit_factor(returns):
    """Sum of positive returns / abs(sum of negative returns)."""
    returns = np.asarray(returns)
    gains = returns[returns > 0].sum()
    losses = -returns[returns < 0].sum()
    if losses == 0:
        return np.inf if gains > 0 else 0.0
    return gains / losses


def quantile_positions(pred, n_q=N_QUANTILES):
    """Map predictions to positions: +1 in top quantile, -1 in bottom, else 0."""
    pred = pd.Series(np.asarray(pred))
    try:
        q = pd.qcut(pred.rank(method="first"), n_q, labels=False)
    except ValueError:
        q = pd.qcut(pred.rank(method="first"), n_q, labels=False, duplicates="drop")
    pos = np.zeros(len(pred))
    pos[q == q.max()] = 1.0    # top quantile -> long
    pos[q == 0] = -1.0         # bottom quantile -> short
    return pos


def strategy_log_returns(pred, realized_logret, n_q=N_QUANTILES):
    """Strategy log return = position * next-day log return (position in {-1,0,+1})."""
    pos = quantile_positions(pred, n_q)
    return pos * np.asarray(realized_logret)


def directional_accuracy(y_true, y_pred):
    return float(np.mean(np.sign(y_pred) == np.sign(y_true)))


def dir_acc_by_quantile(y_true, y_pred, n_q=N_QUANTILES):
    """Directional accuracy within each prediction quantile (useful for trading)."""
    df = pd.DataFrame({"yt": np.asarray(y_true), "yp": np.asarray(y_pred)})
    df["q"] = pd.qcut(df["yp"].rank(method="first"), n_q, labels=False)
    out = df.groupby("q").apply(lambda d: np.mean(np.sign(d["yp"]) == np.sign(d["yt"])))
    out.index = [f"Q{int(i)+1}" for i in out.index]
    return out


def evaluate(y_true, y_pred, realized_logret):
    """Full metric suite for a set of predictions."""
    strat = strategy_log_returns(y_pred, realized_logret)
    simple = np.expm1(strat)           # convert log returns -> simple for Profit Factor
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "ProfitFactor": profit_factor(simple),
        "SpearmanRHO": spearmanr(y_true, y_pred).correlation,
        "DirAcc": directional_accuracy(y_true, y_pred),
    }


print("Evaluation helpers defined.")

In [ ]:
# ---------------------------------------------------------------------------
# Keras MLP wrapped with a small sklearn-style interface
# (MSE loss, L2 weight penalty + dropout regularization, no sample weights)
# ---------------------------------------------------------------------------
class KerasMLPRegressor:
    def __init__(self, units=32, l2=1e-4, dropout=0.2, lr=1e-3, epochs=120, batch_size=32):
        self.units, self.l2, self.dropout = units, l2, dropout
        self.lr, self.epochs, self.batch_size = lr, epochs, batch_size

    def _build(self, input_dim):
        import tensorflow as tf
        from tensorflow import keras
        from tensorflow.keras import layers, regularizers
        tf.random.set_seed(RANDOM_STATE)
        m = keras.Sequential([
            layers.Input(shape=(input_dim,)),
            layers.Dense(self.units, activation="relu",
                         kernel_regularizer=regularizers.l2(self.l2)),
            layers.Dropout(self.dropout),
            layers.Dense(self.units // 2, activation="relu",
                         kernel_regularizer=regularizers.l2(self.l2)),
            layers.Dropout(self.dropout),
            layers.Dense(1, activation="linear"),
        ])
        m.compile(optimizer=keras.optimizers.Adam(self.lr), loss="mse")
        return m

    def fit(self, X, y):
        from tensorflow import keras
        X = np.asarray(X, dtype="float32")
        y = np.asarray(y, dtype="float32")
        self.model_ = self._build(X.shape[1])
        es = keras.callbacks.EarlyStopping(monitor="loss", patience=12, restore_best_weights=True)
        self.history_ = self.model_.fit(X, y, epochs=self.epochs, batch_size=self.batch_size,
                                        verbose=0, callbacks=[es])
        return self

    def predict(self, X):
        X = np.asarray(X, dtype="float32")
        return self.model_.predict(X, verbose=0).ravel()


def make_model(name, params):
    if name == "Ridge":
        return Ridge(**params, random_state=RANDOM_STATE) if "random_state" in Ridge().get_params() else Ridge(**params)
    if name == "RandomForest":
        # n_jobs=1 avoids joblib's parallel pool (keeps the notebook portable
        # across Python builds; the dataset is small so it is still fast).
        return RandomForestRegressor(**params, random_state=RANDOM_STATE, n_jobs=1)
    if name == "LightGBM":
        if HAS_LGBM:
            return LGBMRegressor(**params, random_state=RANDOM_STATE, n_jobs=1, verbosity=-1)
        return GradientBoostingRegressor(**params, random_state=RANDOM_STATE)
    if name == "SVR":
        return SVR(**params)
    if name == "MLP":
        return KerasMLPRegressor(**params)
    raise ValueError(name)


# Hyperparameter grids (kept modest so a full CPU run stays ~5-10 min).
PARAM_GRIDS = {
    "Ridge": [{"alpha": a} for a in (0.1, 1.0, 10.0, 100.0)],
    "RandomForest": [{"n_estimators": ne, "max_depth": md}
                     for ne in (300,) for md in (4, 6, 10)],
    "LightGBM": ([{"n_estimators": ne, "num_leaves": nl, "learning_rate": lr}
                  for ne in (400,) for nl in (15, 31) for lr in (0.02, 0.05)]
                 if HAS_LGBM else
                 [{"n_estimators": ne, "max_depth": md, "learning_rate": lr}
                  for ne in (300,) for md in (2, 3) for lr in (0.02, 0.05)]),
    "SVR": [{"C": c, "gamma": g, "kernel": "rbf"}
            for c in (1.0, 10.0) for g in ("scale", 0.1)],
}
# The Keras MLP (a required model) is included only if TensorFlow is available.
if HAS_TF:
    PARAM_GRIDS["MLP"] = [{"units": u, "l2": 1e-4, "dropout": dr, "lr": 1e-3}
                          for u in (32, 64) for dr in (0.1, 0.3)]
else:
    print("NOTE: TensorFlow missing -> running without the MLP "
          "(4 models). Install tensorflow to include the required neural net.")

print("Model builders and parameter grids ready for:", list(PARAM_GRIDS.keys()))

In [ ]:
from sklearn.metrics import mean_squared_error

tscv = TimeSeriesSplit(n_splits=3)


def cv_score(name, params):
    """Mean validation MSE across TimeSeriesSplit folds of the training set."""
    scores = []
    Xtr_arr, ytr_arr = X_train_s.values, Y_train.values
    for tr_idx, va_idx in tscv.split(Xtr_arr):
        model = make_model(name, params)
        model.fit(Xtr_arr[tr_idx], ytr_arr[tr_idx])
        pred = model.predict(Xtr_arr[va_idx])
        scores.append(mean_squared_error(ytr_arr[va_idx], pred))
    return float(np.mean(scores))


best_models, val_results = {}, {}
for name, grid in PARAM_GRIDS.items():
    best = min(((cv_score(name, p), p) for p in grid), key=lambda t: t[0])
    best_cv_mse, best_params = best
    # Refit best configuration on the full training set, evaluate on validation.
    model = make_model(name, best_params)
    model.fit(X_train_s.values, Y_train.values)
    val_pred = model.predict(X_val_s.values)
    metrics = evaluate(Y_val.values, val_pred, ret_val.values)
    metrics["cv_MSE"] = best_cv_mse
    best_models[name] = {"model": model, "params": best_params}
    val_results[name] = metrics
    print(f"{name:13s} best={best_params} | val MAE={metrics['MAE']:.5f} "
          f"PF={metrics['ProfitFactor']:.3f} RHO={metrics['SpearmanRHO']:.3f} "
          f"DirAcc={metrics['DirAcc']:.3f}")

val_table = pd.DataFrame(val_results).T
val_table

### 6.1 Select the best model

We pick the model with the best **validation Spearman RHO** (ranking quality matters most for a
quantile trading strategy), using Profit Factor and directional accuracy as tie-breakers. The
learning curve of the chosen model is plotted to confirm it learned without severe overfitting
(deliverable: "how well did the model learn").

In [ ]:
# Rank by Spearman RHO (higher is better); fall back gracefully on NaN.
ranking = val_table["SpearmanRHO"].fillna(-np.inf).sort_values(ascending=False)
best_name = ranking.index[0]
best_params = best_models[best_name]["params"]
print("Validation ranking by Spearman RHO:\n", ranking.round(4), "\n")
print(f"==> Selected model: {best_name}  with params {best_params}")

# Compare validation MAE across models.
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
val_table["MAE"].plot(kind="bar", ax=ax[0], color="steelblue", title="Validation MAE (lower better)")
ax[0].set_ylabel("MAE")
val_table["SpearmanRHO"].plot(kind="bar", ax=ax[1], color="seagreen",
                              title="Validation Spearman RHO (higher better)")
ax[1].axhline(0, color="k", lw=0.7)
plt.tight_layout()
plt.savefig(os.path.join(IMG_DIR, "model_comparison.png"), dpi=120)
plt.show()

# If the chosen model is the Keras MLP, show its training loss curve.
chosen = best_models[best_name]["model"]
if best_name == "MLP" and hasattr(chosen, "history_"):
    plt.figure(figsize=(7, 4))
    plt.plot(chosen.history_.history["loss"])
    plt.title("MLP training loss (MSE)")
    plt.xlabel("epoch"); plt.ylabel("loss")
    plt.tight_layout(); plt.show()


# ---------------------------------------------------------------------------
# Learning diagnostics (deliverable: "how well did the model learn?")
#   - per-fold TRAIN vs VALIDATION MSE across TimeSeriesSplit folds
#   - box-and-whisker of validation-fold MSE for every model
# ---------------------------------------------------------------------------
def cv_fold_scores(name, params):
    """Return (train_mse_per_fold, val_mse_per_fold) across TimeSeriesSplit."""
    tr_mses, va_mses = [], []
    Xa, ya = X_train_s.values, Y_train.values
    for tr_idx, va_idx in tscv.split(Xa):
        m = make_model(name, params)
        m.fit(Xa[tr_idx], ya[tr_idx])
        tr_mses.append(mean_squared_error(ya[tr_idx], m.predict(Xa[tr_idx])))
        va_mses.append(mean_squared_error(ya[va_idx], m.predict(Xa[va_idx])))
    return tr_mses, va_mses

fold_val_scores = {}
sel_train, sel_val = None, None
for name in PARAM_GRIDS:
    tr_m, va_m = cv_fold_scores(name, best_models[name]["params"])
    fold_val_scores[name] = va_m
    if name == best_name:
        sel_train, sel_val = tr_m, va_m

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
# Left: box-and-whisker of validation-fold MSE for each model.
ax[0].boxplot(list(fold_val_scores.values()), tick_labels=list(fold_val_scores.keys()))
ax[0].set_title("CV validation-fold MSE by model (box & whisker)")
ax[0].set_ylabel("MSE"); ax[0].tick_params(axis="x", rotation=30)
# Right: train vs validation MSE per fold for the selected model (overfit check).
folds = range(1, len(sel_train) + 1)
ax[1].plot(folds, sel_train, "o-", label="train MSE")
ax[1].plot(folds, sel_val, "s--", label="validation MSE")
ax[1].set_title(f"{best_name}: train vs validation MSE per fold")
ax[1].set_xlabel("TimeSeriesSplit fold"); ax[1].set_ylabel("MSE"); ax[1].legend()
plt.tight_layout()
plt.savefig(os.path.join(IMG_DIR, "learning_diagnostics.png"), dpi=120)
plt.show()

# ---------------------------------------------------------------------------
# Validation directional accuracy BY QUANTILE vs baseline (Project.txt: validation metrics)
# ---------------------------------------------------------------------------
val_pred_best = best_models[best_name]["model"].predict(X_val_s.values)
val_baseline_dir = np.sign(Y_train.mean())
val_baseline_acc = float(np.mean(np.sign(Y_val.values) == val_baseline_dir))
val_qacc = dir_acc_by_quantile(Y_val.values, val_pred_best)
print("Validation directional accuracy by quantile:\n", val_qacc.round(3))
print(f"Validation baseline majority-direction accuracy: {val_baseline_acc:.3f}")

plt.figure(figsize=(8, 4))
val_qacc.plot(kind="bar", color="teal")
plt.axhline(val_baseline_acc, color="red", ls="--", label=f"baseline {val_baseline_acc:.2f}")
plt.axhline(0.5, color="grey", ls=":", label="0.50")
plt.title(f"Validation directional accuracy by quantile ({best_name})")
plt.ylabel("accuracy"); plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(IMG_DIR, "val_dir_acc_by_quantile.png"), dpi=120)
plt.show()

### 6.2 Refit the selected model on train + validation

For the final model we refit on **train + validation** combined (more data, still strictly
before the test period) and freeze it for the one-shot test evaluation.

In [ ]:
X_trval = pd.concat([X_train_s, X_val_s])
Y_trval = pd.concat([Y_train, Y_val])

final_model = make_model(best_name, best_params)
final_model.fit(X_trval.values, Y_trval.values)

test_pred = final_model.predict(X_test_s.values)
print(f"Final model ({best_name}) refit on {len(X_trval)} rows; "
      f"predicted {len(test_pred)} test points.")

## 7. Test-set metrics and directional accuracy by quantile

We report MAE, Profit Factor, Spearman RHO and directional accuracy on the test set, plus
**directional accuracy by prediction quantile** compared with the baseline majority direction.
A good result shows accuracy rising toward the extreme quantiles (Q1/Q5) — which is exactly
what justifies trading only those buckets.

In [ ]:
test_metrics = evaluate(Y_test.values, test_pred, ret_test.values)
print("Test-set metrics:")
for k, v in test_metrics.items():
    print(f"  {k:13s}: {v:.5f}")

# Baseline: always predict the majority direction of the training target.
baseline_dir = np.sign(Y_train.mean())
baseline_acc = float(np.mean(np.sign(Y_test.values) == baseline_dir))

qacc = dir_acc_by_quantile(Y_test.values, test_pred)
print(f"\nBaseline majority-direction accuracy: {baseline_acc:.3f}")
print("\nDirectional accuracy by prediction quantile:")
print(qacc.round(3))

plt.figure(figsize=(8, 4))
qacc.plot(kind="bar", color="darkorange")
plt.axhline(baseline_acc, color="red", ls="--", label=f"baseline {baseline_acc:.2f}")
plt.axhline(0.5, color="grey", ls=":", label="0.50")
plt.title("Test directional accuracy by prediction quantile")
plt.ylabel("accuracy"); plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(IMG_DIR, "dir_acc_by_quantile.png"), dpi=120)
plt.show()

# ---------------------------------------------------------------------------
# Combined TRAIN / VALIDATION / TEST metric table (deliverable l: train & test metrics).
#   Train & validation use the train-only model; test uses the train+val model.
#   Train metrics are in-sample (optimistic) and labelled as such.
# ---------------------------------------------------------------------------
train_model = best_models[best_name]["model"]      # fit on training data only
train_pred = train_model.predict(X_train_s.values)
val_pred_b = train_model.predict(X_val_s.values)

metrics_table = pd.DataFrame({
    "Train (in-sample)": evaluate(Y_train.values, train_pred, ret_train.values),
    "Validation":        evaluate(Y_val.values,   val_pred_b, ret_val.values),
    "Test":              test_metrics,
}).T
print("\nStatistical + financial metrics across splits:")
metrics_table.round(5)

## 8. Feature importance — SHAP (test set)

We explain the final model's test-set predictions with **SHAP** values. For tree models we use
the fast `TreeExplainer`; otherwise a sampled explainer. If SHAP is unavailable, we fall back to
sklearn permutation importance so the notebook still runs.

In [ ]:
feat_names = list(X_test_s.columns)
importance = None

if HAS_SHAP:
    try:
        est = final_model
        if best_name in ("RandomForest", "LightGBM"):
            # TreeExplainer supports RF, LightGBM and the GradientBoosting fallback.
            explainer = shap.TreeExplainer(est)
            shap_vals = explainer.shap_values(X_test_s.values)
        elif best_name == "Ridge":
            explainer = shap.LinearExplainer(est, X_trval.values)
            shap_vals = explainer.shap_values(X_test_s.values)
        else:
            # SVR / MLP: KernelExplainer on a small background + sample (kept small for speed)
            bg = shap.sample(X_trval.values, 50, random_state=RANDOM_STATE)
            f = (lambda d: final_model.predict(d))
            explainer = shap.KernelExplainer(f, bg)
            sample = X_test_s.values[:100]
            shap_vals = explainer.shap_values(sample, nsamples=100)
        importance = pd.Series(np.abs(np.array(shap_vals)).mean(axis=0),
                               index=feat_names).sort_values(ascending=False)
        plt.figure(figsize=(8, 5))
        importance.plot(kind="barh", color="purple")
        plt.gca().invert_yaxis()
        plt.title(f"Mean |SHAP value| — {best_name} (test set)")
        plt.tight_layout()
        plt.savefig(os.path.join(IMG_DIR, "shap_importance.png"), dpi=120)
        plt.show()
    except Exception as e:
        print("SHAP failed, using permutation importance instead:", e)
        importance = None

if importance is None:
    from sklearn.inspection import permutation_importance
    r = permutation_importance(final_model, X_test_s.values, Y_test.values,
                               n_repeats=10, random_state=RANDOM_STATE,
                               scoring="neg_mean_squared_error")
    importance = pd.Series(r.importances_mean, index=feat_names).sort_values(ascending=False)
    plt.figure(figsize=(8, 5))
    importance.plot(kind="barh", color="purple")
    plt.gca().invert_yaxis()
    plt.title(f"Permutation importance — {best_name} (test set)")
    plt.tight_layout()
    plt.savefig(os.path.join(IMG_DIR, "perm_importance.png"), dpi=120)
    plt.show()

print("Top features:\n", importance.head(8).round(5))

## 9. Trading strategy and equity curve

**Strategy (decision #10):** each day rank the model's predicted next-day move into 5 quantiles.
Go **long** in the top quantile (Q5), **short** in the bottom quantile (Q1), and stay **flat**
otherwise. We compare the **test equity curve** against **buy-and-hold**.

Because the target is a log return, the equity curve uses log-return math
(`cumsum` then `exp`); strategy log returns are `position × next-day log return`.

In [ ]:
def equity_from_pred(pred, realized_logret):
    """Strategy & buy-and-hold equity curves (growth of $1) from predictions."""
    pos = quantile_positions(pred)
    strat = pos * np.asarray(realized_logret)
    return pos, np.exp(np.cumsum(strat)), np.exp(np.cumsum(np.asarray(realized_logret)))

# Train and validation use the train-only model (in-sample / out-of-sample);
# test uses the final train+val model.
train_pos, train_eq, train_bh = equity_from_pred(train_model.predict(X_train_s.values), ret_train.values)
val_pos,   val_eq,   val_bh   = equity_from_pred(train_model.predict(X_val_s.values),   ret_val.values)
test_pos,  test_eq,  test_bh  = equity_from_pred(test_pred, ret_test.values)
strat_logret = test_pos * ret_test.values            # used by the diagnostics cell

# Deliverable k: TRAIN and TEST equity curves (validation shown too for completeness).
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
for a, (name, idx, eq, bh) in zip(ax, [
        ("Train (in-sample)", ret_train.index, train_eq, train_bh),
        ("Validation",        ret_val.index,   val_eq,   val_bh),
        ("Test",              ret_test.index,  test_eq,  test_bh)]):
    a.plot(idx, eq, label="Quantile strategy", color="navy", lw=1.4)
    a.plot(idx, bh, label="Buy & hold", color="grey", lw=1.1, ls="--")
    a.set_title(f"{TARGET} {name} equity"); a.set_ylabel("growth of $1"); a.legend()
plt.tight_layout()
plt.savefig(os.path.join(IMG_DIR, "equity_curves_train_val_test.png"), dpi=120)
plt.show()

n_long = int((test_pos > 0).sum())
n_short = int((test_pos < 0).sum())
n_flat = int((test_pos == 0).sum())
print(f"Positions on test set -> long: {n_long}, short: {n_short}, flat: {n_flat}")

## 10. Equity-curve diagnostics

- **Sharpe ratio, Profit Factor, CAGR** of the test strategy.
- **White reality check** (bootstrap) p-value — adapted for log returns.
- **Monte Carlo permutation** p-value using **Profit Factor** as the metric.

> Per the course note, the strategy need not be profitable — the *logic* must be correct and the
> significance tests honestly reported.

In [ ]:
TRADING_DAYS = 252

def sharpe_ratio(logret, periods=TRADING_DAYS):
    logret = np.asarray(logret)
    sd = logret.std(ddof=1)
    return np.sqrt(periods) * logret.mean() / sd if sd > 0 else 0.0

def cagr(logret, periods=TRADING_DAYS):
    total = np.exp(np.sum(logret))           # final growth multiple
    years = len(logret) / periods
    return total ** (1 / years) - 1 if years > 0 else 0.0

strat_simple = np.expm1(strat_logret)        # simple returns for Profit Factor
pf_test = profit_factor(strat_simple)
sharpe_test = sharpe_ratio(strat_logret)
cagr_test = cagr(strat_logret)
print(f"Test strategy  ->  Sharpe: {sharpe_test:.3f} | Profit Factor: {pf_test:.3f} | CAGR: {cagr_test*100:.2f}%")

# ---------------------------------------------------------------------------
# White reality check (bootstrap on the strategy's mean log return)
# ---------------------------------------------------------------------------
def white_reality_check(strat_logret, n_boot=2000, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    x = np.asarray(strat_logret)
    demeaned = x - x.mean()                   # H0: true mean = 0
    obs = x.mean()
    boot_means = np.array([rng.choice(demeaned, size=len(x), replace=True).mean()
                           for _ in range(n_boot)])
    return float(np.mean(boot_means >= obs))

# ---------------------------------------------------------------------------
# Monte Carlo permutation test (shuffle positions vs returns; metric = Profit Factor)
# ---------------------------------------------------------------------------
def monte_carlo_pf(positions, realized_logret, n_perm=2000, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    obs_pf = profit_factor(np.expm1(positions * realized_logret))
    obs_pf = obs_pf if np.isfinite(obs_pf) else 1e9
    count = 0
    pos = np.asarray(positions)
    for _ in range(n_perm):
        shuffled = rng.permutation(pos)
        pf = profit_factor(np.expm1(shuffled * realized_logret))
        pf = pf if np.isfinite(pf) else 1e9
        if pf >= obs_pf:
            count += 1
    return float(count / n_perm)

p_white = white_reality_check(strat_logret)
p_mc = monte_carlo_pf(test_pos, ret_test.values)
print(f"White reality check p-value     : {p_white:.4f}")
print(f"Monte Carlo permutation p-value : {p_mc:.4f}")
print("\n(Lower p-values => the result is less likely to be due to chance.)")

## 11. Summary

| Item | Result |
|------|--------|
| Target | FLEX next-day log return (daily, single-target) |
| Indicators | 18 (12 external), worst 4 discarded by mutual information |
| Models tried | Ridge, Random Forest, LightGBM, SVR, Keras MLP |
| Best model | *(see selection above)* |
| Test metrics | MAE / Profit Factor / Spearman RHO / directional accuracy (above) |
| Trading | Long Q5 / short Q1 / flat otherwise |
| Significance | White reality check + Monte Carlo permutation p-values (above) |

**Honesty note (per course rules):** the strategy is not guaranteed to be profitable; the goal is
a correct, leak-free pipeline (causal scaling, `TimeSeriesSplit`, feature selection on training
data only, a held-out test set evaluated once) with the significance of any edge reported
transparently.

### Notes for the presentation (explicit answers to spec items)
- **Seed program (items a, c):** none — this notebook was built **from scratch**, so there are no
  "improvements over a seed program" to list.
- **Input selection vs extraction (item f):** we use **feature selection** (`SelectKBest`, mutual
  information) rather than **PCA extraction**. PCA was considered but rejected because we want to
  keep features **interpretable** for SHAP and the trading narrative, and tree models (RF/LightGBM)
  do not benefit from decorrelated inputs. PCA would only be warranted for a model that needs
  uncorrelated inputs (e.g. Naive Bayes / VAR), which we do not use.
- **Neural-network specifics:** loss = **MSE**; regularization = **L2 weight penalty + dropout**;
  **no sample weights** were used in training.
- **Option (item p):** see `../project_decisions.txt`; the course "Option" list is still
  instructor-defined — confirm the official option number before submission.

### Possible extensions
- Add an LSTM/Transformer sequence model (Keras) for comparison.
- Add transaction costs / slippage to the equity curve.
- Try the multi-target version across the mid-cap universe with Alphalens evaluation.

## 12. Environment listing (deliverable 1: conda-list)

Writes the exact package versions used into `conda-list.txt` for submission. Tries `conda list`
first; if conda is not on PATH, falls back to `pip freeze` (equivalent reproducibility info).

In [ ]:
import subprocess
import sys

listing, source = None, None
try:
    listing = subprocess.check_output(["conda", "list"], text=True, stderr=subprocess.DEVNULL)
    source = "conda list"
except Exception:
    try:
        listing = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
        source = "pip freeze (conda not found on PATH)"
    except Exception as e:
        listing = f"Could not generate environment listing: {e}"
        source = "error"

with open("conda-list.txt", "w", encoding="utf-8") as f:
    f.write(f"# Environment listing via {source}\n")
    f.write(f"# Python {sys.version}\n\n")
    f.write(listing)

print(f"Wrote conda-list.txt via '{source}'.")
print("First lines:\n", "\n".join(listing.splitlines()[:12]))